In [147]:
import os
import torch
torch.manual_seed(12345)
import torch.nn.functional as F
from torch.nn import Linear
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
import networkx as nx
from torch_geometric.data import Data
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import NormalizeFeatures

In [148]:
dataset=Planetoid(root='data/Planetoid',name='Cora',transform=NormalizeFeatures())

In [149]:
dataset

Cora()

In [150]:
data=dataset[0]

In [151]:
data

Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])

In [152]:
data.x.shape

torch.Size([2708, 1433])

In [252]:
data.num_nodes

2708

In [153]:
data.x[0]

tensor([0., 0., 0.,  ..., 0., 0., 0.])

In [154]:
data.num_nodes

2708

In [255]:
# sum=0
# for i in range(data.num_nodes):
#     sum+=data.x[i]
# mean=sum/data.num_nodes
# mean

In [262]:
mean=data.x.mean(dim=0)
mean

tensor([0.0059, 0.0122, 0.0258,  ..., 0.0022, 0.0240, 0.0044])

In [263]:
std = data.x.std(dim=0)
std

tensor([0.0767, 0.1097, 0.1587,  ..., 0.0470, 0.1531, 0.0664])

gaussian membership

In [264]:
T_gaussian = torch.exp(-((data.x - mean)**2) / (2 * std**2))
T_gaussian.shape

torch.Size([2708, 1433])

In [291]:
T_gaussian

tensor([[0.9970, 0.9939, 0.9868,  ..., 0.9989, 0.9878, 0.9978],
        [0.9970, 0.9939, 0.9868,  ..., 0.9989, 0.9878, 0.9978],
        [0.9970, 0.9939, 0.9868,  ..., 0.9989, 0.9878, 0.9978],
        ...,
        [0.9970, 0.9939, 0.9868,  ..., 0.9989, 0.9878, 0.9978],
        [0.9970, 0.9939, 0.9868,  ..., 0.9989, 0.9878, 0.9978],
        [0.9970, 0.9939, 0.9868,  ..., 0.9989, 0.9878, 0.9978]])

In [292]:
data.x

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])

In [293]:
val = torch.exp(-((data.x[0][0] - mean[0])**2) / (2 * std[0]**2))
print(val)

tensor(0.9970)


hyperparameters

In [294]:
alp=0.5
beta=2.0

In [297]:
F_gaussian=1-torch.exp(-((data.x - mean)**2) / (2 * (alp*std)**2))
F_gaussian

tensor([[0.0118, 0.0244, 0.0517,  ..., 0.0044, 0.0480, 0.0089],
        [0.0118, 0.0244, 0.0517,  ..., 0.0044, 0.0480, 0.0089],
        [0.0118, 0.0244, 0.0517,  ..., 0.0044, 0.0480, 0.0089],
        ...,
        [0.0118, 0.0244, 0.0517,  ..., 0.0044, 0.0480, 0.0089],
        [0.0118, 0.0244, 0.0517,  ..., 0.0044, 0.0480, 0.0089],
        [0.0118, 0.0244, 0.0517,  ..., 0.0044, 0.0480, 0.0089]])

In [299]:
I_gaussian=1-torch.exp(-((data.x - mean)**2) / (2 * (beta*std)**2))
I_gaussian

tensor([[0.0007, 0.0015, 0.0033,  ..., 0.0003, 0.0031, 0.0006],
        [0.0007, 0.0015, 0.0033,  ..., 0.0003, 0.0031, 0.0006],
        [0.0007, 0.0015, 0.0033,  ..., 0.0003, 0.0031, 0.0006],
        ...,
        [0.0007, 0.0015, 0.0033,  ..., 0.0003, 0.0031, 0.0006],
        [0.0007, 0.0015, 0.0033,  ..., 0.0003, 0.0031, 0.0006],
        [0.0007, 0.0015, 0.0033,  ..., 0.0003, 0.0031, 0.0006]])

In [164]:
T_gaussian+I_gaussian+F_gaussian

tensor([[1.0095, 1.0191, 1.0402,  ..., 1.0026, 1.0371, 1.0070],
        [1.0095, 1.0191, 1.0402,  ..., 1.0026, 1.0371, 1.0070],
        [1.0095, 1.0191, 1.0402,  ..., 1.0026, 1.0371, 1.0070],
        ...,
        [1.0095, 1.0191, 1.0402,  ..., 1.0026, 1.0371, 1.0070],
        [1.0095, 1.0191, 1.0402,  ..., 1.0026, 1.0371, 1.0070],
        [1.0095, 1.0191, 1.0402,  ..., 1.0026, 1.0371, 1.0070]])

In [165]:
print(T_gaussian,'\n',I_gaussian,'\n',F_gaussian)

tensor([[0.9971, 0.9941, 0.9873,  ..., 0.9992, 0.9883, 0.9979],
        [0.9971, 0.9941, 0.9873,  ..., 0.9992, 0.9883, 0.9979],
        [0.9971, 0.9941, 0.9873,  ..., 0.9992, 0.9883, 0.9979],
        ...,
        [0.9971, 0.9941, 0.9873,  ..., 0.9992, 0.9883, 0.9979],
        [0.9971, 0.9941, 0.9873,  ..., 0.9992, 0.9883, 0.9979],
        [0.9971, 0.9941, 0.9873,  ..., 0.9992, 0.9883, 0.9979]]) 
 tensor([[0.0007, 0.0015, 0.0032,  ..., 0.0002, 0.0029, 0.0005],
        [0.0007, 0.0015, 0.0032,  ..., 0.0002, 0.0029, 0.0005],
        [0.0007, 0.0015, 0.0032,  ..., 0.0002, 0.0029, 0.0005],
        ...,
        [0.0007, 0.0015, 0.0032,  ..., 0.0002, 0.0029, 0.0005],
        [0.0007, 0.0015, 0.0032,  ..., 0.0002, 0.0029, 0.0005],
        [0.0007, 0.0015, 0.0032,  ..., 0.0002, 0.0029, 0.0005]]) 
 tensor([[0.0117, 0.0236, 0.0497,  ..., 0.0032, 0.0459, 0.0086],
        [0.0117, 0.0236, 0.0497,  ..., 0.0032, 0.0459, 0.0086],
        [0.0117, 0.0236, 0.0497,  ..., 0.0032, 0.0459, 0.0086],
        

In [166]:
mean

tensor([0.0003, 0.0006, 0.0013,  ..., 0.0001, 0.0012, 0.0002])

In [ ]:

num_features = dataset.num_node_features
num_classes = dataset.num_classes


class GNN_Embedder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        
    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        h = F.normalize(x, dim=1)  # normalize embeddings for angular fuzzification
        return h

hidden_dim = 16
model = GNN_Embedder(num_features, hidden_dim).to(device)

# ----------------------------
# 3️⃣ Define Class Prototypes as nn.Parameter
# ----------------------------
W = torch.nn.Parameter(torch.randn(num_classes, hidden_dim, device=device))

# ----------------------------
# 4️⃣ Angular Fuzzification Function
# ----------------------------
def angular_fuzzification(h, W):
    """
    h: node embeddings [num_nodes, hidden_dim]
    W: class prototypes [num_classes, hidden_dim]
    returns: T, F, I for each node-class pair
    """
    W_norm = F.normalize(W, dim=1)  # normalize prototypes for cosine similarity
    cos_sim = torch.matmul(h, W_norm.T)
    T = (1 + cos_sim) / 2
    F_val = (1 - cos_sim) / 2
    I = 1 - torch.abs(cos_sim)
    return T, F_val, I

# ----------------------------
# 5️⃣ Loss Function
# ----------------------------
def prototype_loss(T, y, mask):
    T_masked = T[mask]
    y_masked = y[mask]
    loss = F.cross_entropy(T_masked, y_masked)
    return loss

# ----------------------------
# 6️⃣ Optimizer
# ----------------------------
optimizer = torch.optim.Adam(list(model.parameters()) + [W], lr=0.01, weight_decay=5e-4)

# ----------------------------
# 7️⃣ Training Loop
# ----------------------------
num_epochs = 200
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    
    h = model(data.x, data.edge_index)  # node embeddings
    
    # Angular fuzzification
    T, F_val, I = angular_fuzzification(h, W)
    
    loss = prototype_loss(T, data.y, data.train_mask)
    loss.backward()
    optimizer.step()
    
    # Optional: in-place normalize prototypes after each step
    with torch.no_grad():
        W.copy_(F.normalize(W, dim=1))
    
    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

# ----------------------------
# 8️⃣ Evaluation
# ----------------------------
model.eval()
with torch.no_grad():
    h = model(data.x, data.edge_index)
    T, F_val, I = angular_fuzzification(h, W)
    
    pred = T.argmax(dim=1)
    correct = (pred[data.test_mask] == data.y[data.test_mask]).sum()
    acc = int(correct) / int(data.test_mask.sum())
    print(f"Test Accuracy: {acc:.4f}")

# ----------------------------
# 9️⃣ Access T, F, I values (example)
# ----------------------------
print("Truth T for first 5 nodes:\n", T[:5])
print("Falsity F for first 5 nodes:\n", F_val[:5])
print("Indeterminacy I for first 5 nodes:\n", I[:5])

Epoch 0, Loss: 1.9568
Epoch 20, Loss: 1.4883
Epoch 40, Loss: 1.4744
Epoch 60, Loss: 1.4710
Epoch 80, Loss: 1.4720
Epoch 100, Loss: 1.4700
Epoch 120, Loss: 1.4705
Epoch 140, Loss: 1.4700
Epoch 160, Loss: 1.4706
Epoch 180, Loss: 1.4699
Test Accuracy: 0.7870
Truth T for first 5 nodes:
 tensor([[0.4090, 0.4262, 0.4176, 0.9995, 0.4139, 0.4283, 0.4063],
        [0.4205, 0.4224, 0.4020, 0.3995, 0.9991, 0.4302, 0.4233],
        [0.4123, 0.4247, 0.4029, 0.4144, 0.9992, 0.4255, 0.4179],
        [0.9986, 0.4343, 0.4219, 0.3978, 0.4056, 0.4466, 0.4013],
        [0.4060, 0.4270, 0.4219, 0.9992, 0.4197, 0.4223, 0.4046]])
Falsity F for first 5 nodes:
 tensor([[5.9102e-01, 5.7379e-01, 5.8241e-01, 4.5571e-04, 5.8610e-01, 5.7174e-01,
         5.9367e-01],
        [5.7946e-01, 5.7761e-01, 5.9799e-01, 6.0051e-01, 9.3681e-04, 5.6984e-01,
         5.7669e-01],
        [5.8765e-01, 5.7526e-01, 5.9705e-01, 5.8561e-01, 8.2102e-04, 5.7454e-01,
         5.8215e-01],
        [1.3787e-03, 5.6571e-01, 5.7814e-01, 6

In [233]:
W = torch.nn.Parameter(torch.randn(num_classes, hidden_dim, device=device))

In [ ]:
#angular fuzzification
def angular_fuzzification(h, W):
    """
    h: node embeddings [num_nodes, hidden_dim]
    W: class prototypes [num_classes, hidden_dim]
    returns: T, F, I for each node-class pair
    """
    W_norm = F.normalize(W, dim=1)  # normalize prototypes for cosine similarity
    cos_sim = torch.matmul(h, W_norm.T)
    T = (1 + cos_sim) / 2
    F_val = (1 - cos_sim) / 2
    I = 1 - torch.abs(cos_sim)
    return T, F_val, I

#loss fucntuion
def prototype_loss(T, y, mask):
    T_masked = T[mask]
    y_masked = y[mask]
    loss = F.cross_entropy(T_masked, y_masked)
    return loss

#optimiser
optimizer = torch.optim.Adam(list(model.parameters()) + [W], lr=0.01, weight_decay=5e-4)

In [235]:
num_epochs = 200
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    
    h = model(data.x, data.edge_index)  # node embeddings
    
    # Angular fuzzification
    T, F_val, I = angular_fuzzification(h, W)
    
    loss = prototype_loss(T, data.y, data.train_mask)
    loss.backward()
    optimizer.step()
    
    # Optional: in-place normalize prototypes after each step
    with torch.no_grad():
        W.copy_(F.normalize(W, dim=1))
    
    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

#evaluation
model.eval()
with torch.no_grad():
    h = model(data.x, data.edge_index)
    T, F_val, I = angular_fuzzification(h, W)
    
    pred = T.argmax(dim=1)
    correct = (pred[data.test_mask] == data.y[data.test_mask]).sum()
    acc = int(correct) / int(data.test_mask.sum())
    print(f"Test Accuracy: {acc:.4f}")


Epoch 0, Loss: 1.9758
Epoch 20, Loss: 1.4961
Epoch 40, Loss: 1.4770
Epoch 60, Loss: 1.4732
Epoch 80, Loss: 1.4720
Epoch 100, Loss: 1.4703
Epoch 120, Loss: 1.4731
Epoch 140, Loss: 1.4701
Epoch 160, Loss: 1.4699
Epoch 180, Loss: 1.4710
Test Accuracy: 0.7800


In [236]:
print("Truth T for first 5 nodes:\n", T[:5])
print("Falsity F for first 5 nodes:\n", F_val[:5])
print("Indeterminacy I for first 5 nodes:\n", I[:5])

Truth T for first 5 nodes:
 tensor([[0.4139, 0.4264, 0.4065, 0.9997, 0.4186, 0.4157, 0.4177],
        [0.4149, 0.4216, 0.4027, 0.4161, 0.9999, 0.4254, 0.4201],
        [0.4215, 0.4120, 0.4032, 0.4292, 0.9998, 0.4197, 0.4153],
        [0.9998, 0.4165, 0.4171, 0.4156, 0.4128, 0.4185, 0.4188],
        [0.4090, 0.4273, 0.4094, 0.9993, 0.4201, 0.4162, 0.4174]])
Falsity F for first 5 nodes:
 tensor([[5.8607e-01, 5.7360e-01, 5.9347e-01, 2.5368e-04, 5.8136e-01, 5.8428e-01,
         5.8233e-01],
        [5.8512e-01, 5.7837e-01, 5.9729e-01, 5.8390e-01, 9.1255e-05, 5.7459e-01,
         5.7991e-01],
        [5.7853e-01, 5.8797e-01, 5.9678e-01, 5.7080e-01, 2.1780e-04, 5.8027e-01,
         5.8468e-01],
        [2.3550e-04, 5.8350e-01, 5.8292e-01, 5.8436e-01, 5.8722e-01, 5.8148e-01,
         5.8120e-01],
        [5.9099e-01, 5.7271e-01, 5.9062e-01, 7.0813e-04, 5.7994e-01, 5.8384e-01,
         5.8263e-01]])
Indeterminacy I for first 5 nodes:
 tensor([[8.2786e-01, 8.5280e-01, 8.1305e-01, 5.0735e-04, 8.

In [238]:
data

Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])

In [278]:
data.x

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])

In [279]:
mean[:5]

tensor([0.0059, 0.0122, 0.0258, 0.0266, 0.0609])

In [280]:
std[:5]

tensor([0.0767, 0.1097, 0.1587, 0.1609, 0.2392])

In [281]:
T_gaussian[0][0]

tensor(0.9970)

In [282]:
T_angular[0][0]

tensor(0.4254, grad_fn=<SelectBackward0>)

In [283]:
T_gaussian.shape

torch.Size([2708, 1433])

In [284]:
data.x

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])

In [285]:
T_gaussian.dtype

torch.float32

In [286]:
1433*3

4299

In [300]:
combined = torch.stack([T_gaussian, F_gaussian, I_gaussian], dim=2)

In [301]:
combined

tensor([[[9.9703e-01, 1.1812e-02, 7.4238e-04],
         [9.9385e-01, 2.4362e-02, 1.5403e-03],
         [9.8682e-01, 5.1668e-02, 3.3102e-03],
         ...,
         [9.9889e-01, 4.4297e-03, 2.7746e-04],
         [9.8778e-01, 4.7979e-02, 3.0683e-03],
         [9.9778e-01, 8.8593e-03, 5.5599e-04]],

        [[9.9703e-01, 1.1812e-02, 7.4238e-04],
         [9.9385e-01, 2.4362e-02, 1.5403e-03],
         [9.8682e-01, 5.1668e-02, 3.3102e-03],
         ...,
         [9.9889e-01, 4.4297e-03, 2.7746e-04],
         [9.8778e-01, 4.7979e-02, 3.0683e-03],
         [9.9778e-01, 8.8593e-03, 5.5599e-04]],

        [[9.9703e-01, 1.1812e-02, 7.4238e-04],
         [9.9385e-01, 2.4362e-02, 1.5403e-03],
         [9.8682e-01, 5.1668e-02, 3.3102e-03],
         ...,
         [9.9889e-01, 4.4297e-03, 2.7746e-04],
         [9.8778e-01, 4.7979e-02, 3.0683e-03],
         [9.9778e-01, 8.8593e-03, 5.5599e-04]],

        ...,

        [[9.9703e-01, 1.1812e-02, 7.4238e-04],
         [9.9385e-01, 2.4362e-02, 1.5403e-03]

In [302]:
combined.shape

torch.Size([2708, 1433, 3])

In [303]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv

# =========================
# Load Cora Dataset
# =========================
dataset = Planetoid(root='/tmp/Cora', name='Cora')
data = dataset[0]

# =========================
# Neutrosophic Fuzzification
# =========================
class NeutrosophicFuzzification(nn.Module):
    def __init__(self, alpha=0.5, beta=2.0):
        super().__init__()
        self.alpha = alpha
        self.beta = beta

    def forward(self, x):
        mean = x.mean(dim=0, keepdim=True)
        std = x.std(dim=0, keepdim=True)

        std[std < 1e-6] = 1e-6  # stability

        # Gaussian T, F, I
        T = torch.exp(-((x - mean) ** 2) / (2 * std ** 2))
        F_val = 1 - torch.exp(-((x - mean) ** 2) / (2 * (self.alpha * std) ** 2))
        I = 1 - torch.exp(-((x - mean) ** 2) / (2 * (self.beta * std) ** 2))

        # Concatenate → [N, 3F]
        return torch.cat([T, I, F_val], dim=1)


# =========================
# First-Order TSK Rule Layer
# =========================
class TSKRuleLayer(nn.Module):
    def __init__(self, in_dim, num_rules, out_dim):
        super().__init__()

        self.num_rules = num_rules

        # Rule parameters
        self.w = nn.Parameter(torch.randn(num_rules, in_dim))
        self.b = nn.Parameter(torch.zeros(num_rules))

        # Consequent parameters (first-order)
        self.a = nn.Parameter(torch.randn(num_rules, in_dim, out_dim))
        self.c = nn.Parameter(torch.zeros(num_rules, out_dim))

    def forward(self, x):
        # x: [N, D]

        # Rule firing strengths
        r = torch.matmul(x, self.w.T) + self.b  # [N, K]
        r_hat = F.softmax(r, dim=1)             # normalize

        # First-order TSK output
        outputs = []
        for k in range(self.num_rules):
            y_k = torch.matmul(x, self.a[k]) + self.c[k]  # [N, out_dim]
            outputs.append(y_k)

        outputs = torch.stack(outputs, dim=1)  # [N, K, out_dim]

        # Weighted sum (defuzzification)
        r_hat = r_hat.unsqueeze(-1)  # [N, K, 1]
        y = torch.sum(r_hat * outputs, dim=1)  # [N, out_dim]

        return y


# =========================
# N-GNN Model
# =========================
class NGNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()

        self.fuzz = NeutrosophicFuzzification()

        # GNN layers
        self.conv1 = GCNConv(input_dim * 3, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)

        # Rule layer
        self.tsk = TSKRuleLayer(hidden_dim, num_rules=5, out_dim=num_classes)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        # Step 1: Neutrosophic fuzzification
        x = self.fuzz(x)

        # Step 2: GNN propagation
        h = F.relu(self.conv1(x, edge_index))
        h = F.relu(self.conv2(h, edge_index))

        # Step 3: Rule layer (TSK)
        out = self.tsk(h)

        return out


# =========================
# Training Setup
# =========================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = NGNN(
    input_dim=dataset.num_node_features,
    hidden_dim=64,
    num_classes=dataset.num_classes
).to(device)

data = data.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)


# =========================
# Training Loop
# =========================
def train():
    model.train()
    optimizer.zero_grad()

    out = model(data)

    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()

    return loss.item()


# =========================
# Evaluation
# =========================
@torch.no_grad()
def test():
    model.eval()
    out = model(data)

    pred = out.argmax(dim=1)

    accs = []
    for mask in [data.train_mask, data.val_mask, data.test_mask]:
        acc = (pred[mask] == data.y[mask]).sum().item() / mask.sum().item()
        accs.append(acc)

    return accs


# =========================
# Run Training
# =========================
for epoch in range(201):
    loss = train()
    train_acc, val_acc, test_acc = test()

    if epoch % 20 == 0:
        print(f"Epoch {epoch:03d} | Loss: {loss:.4f} | "
              f"Train: {train_acc:.4f} | Val: {val_acc:.4f} | Test: {test_acc:.4f}")

Processing...
Done!


Epoch 000 | Loss: 2.9604 | Train: 0.1429 | Val: 0.1620 | Test: 0.1490
Epoch 020 | Loss: 12.0079 | Train: 0.1429 | Val: 0.1560 | Test: 0.1440
Epoch 040 | Loss: 1.5176 | Train: 0.4714 | Val: 0.4640 | Test: 0.4750
Epoch 060 | Loss: 0.5467 | Train: 0.8571 | Val: 0.5100 | Test: 0.5490
Epoch 080 | Loss: 0.3115 | Train: 0.9786 | Val: 0.6620 | Test: 0.6960
Epoch 100 | Loss: 0.2250 | Train: 0.9929 | Val: 0.6820 | Test: 0.7000
Epoch 120 | Loss: 0.1675 | Train: 1.0000 | Val: 0.6760 | Test: 0.6940
Epoch 140 | Loss: 0.1247 | Train: 1.0000 | Val: 0.6760 | Test: 0.7010
Epoch 160 | Loss: 0.0984 | Train: 1.0000 | Val: 0.6840 | Test: 0.7060
Epoch 180 | Loss: 0.0801 | Train: 1.0000 | Val: 0.7020 | Test: 0.7130
Epoch 200 | Loss: 0.0671 | Train: 1.0000 | Val: 0.7020 | Test: 0.7170


In [5]:
#!/usr/bin/env python3
"""
═══════════════════════════════════════════════════════════════════════════════
  Neutrosophic Graph Neural Network (N-GNN) for Node Classification
  on Citation Networks (Cora Dataset)
═══════════════════════════════════════════════════════════════════════════════

  This module implements an N-GNN that explicitly models three dimensions
  of uncertainty using neutrosophic logic:
    - Truth (T):           degree to which a feature value is typical
    - Indeterminacy (I):   degree of ambiguity or undecidedness
    - Falsity (F):         degree to which a feature value is atypical

  Architecture (per layer):
  ─────────────────────────────────────────────────────────────────────────
  ┌──────────────┐    ┌────────────────┐    ┌─────────────┐    ┌────────┐
  │  Input Graph │ →  │ Neutrosophic   │ →  │ TSK Rule    │ →  │ Defuzz │
  │  G = (V, E)  │    │ Fuzzification  │    │ Aggregation │    │ + Res  │
  │  X_V, X_E    │    │ (T, I, F)      │    │ K rules     │    │ + ReLU │
  └──────────────┘    └────────────────┘    └─────────────┘    └────────┘

  Multi-layer stacking with residual connections:
    H^(l) = σ( f^(l)(H^(l-1), A) + H^(l-1) )

  References
  ──────────
  [1] Meenakshi et al., "Advanced risk prediction in healthcare:
      Neutrosophic Graph Neural Networks for disease transmission",
      Complex & Intelligent Systems (2025) 11:413.
  [2] Fujita, "Superhypergraph neural networks and plithogenic graph
      neural networks: Theoretical foundations", arXiv 2024.
  [3] Kaviyarasu et al., "The connectivity indices concept of neutrosophic
      graph", Scientific Reports (2024) 14:4891.

  Usage
  ─────
    python ngnn_node_classification.py

  Outputs (saved to ./outputs/):
    - ngnn_architecture.png       Architecture diagram
    - ngnn_fuzzification.png      Gaussian T/I/F membership functions
    - ngnn_training_curves.png    Loss and accuracy over epochs
    - ngnn_ablation_bar.png       Ablation comparison bar chart
    - ngnn_alpha_beta_heatmap.png Hyperparameter sensitivity heatmap
    - ngnn_summary.json           Numerical results summary
"""

# ═══════════════════════════════════════════════════════════════════════════════
# IMPORTS
# ═══════════════════════════════════════════════════════════════════════════════

import os
import copy
import gc
import json
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.datasets import Planetoid
from torch_geometric.utils import add_self_loops, degree
from sklearn.decomposition import PCA
from sklearn.metrics import f1_score, classification_report

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# OUT_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), "outputs")
OUT_DIR = os.path.join(os.getcwd(), "outputs")
os.makedirs(OUT_DIR, exist_ok=True)


# ═══════════════════════════════════════════════════════════════════════════════
# §1  DATA LOADING
# ═══════════════════════════════════════════════════════════════════════════════

def load_cora(root="./data", reduce_dim=64):
    """
    Load the Cora citation network dataset.

    Attempts to download via PyTorch Geometric's Planetoid loader.  If
    network access is unavailable, generates a community-structured
    synthetic graph that mirrors Cora's statistics:
      2 708 nodes · 7 classes · sparse binary features · ~10 500 edges

    Parameters
    ----------
    root : str
        Directory for dataset caching.
    reduce_dim : int or None
        If set, apply PCA to reduce feature dimensionality (saves memory
        and speeds up training with negligible accuracy loss).

    Returns
    -------
    data : torch_geometric.data.Data
    num_classes : int
    """
    try:
        dataset = Planetoid(root=root, name="Cora")
        data = dataset[0]
        num_classes = dataset.num_classes
        source = "Planetoid"
    except Exception:
        print("  [info] Planetoid download unavailable — generating "
              "community-structured synthetic Cora replica")
        data, num_classes = _synthetic_cora()
        source = "Synthetic"

    if reduce_dim and data.x.shape[1] > reduce_dim:
        pca = PCA(n_components=reduce_dim)
        x_red = pca.fit_transform(data.x.cpu().numpy()).astype(np.float32)
        var_kept = pca.explained_variance_ratio_.sum() * 100
        data.x = torch.tensor(x_red)
        print(f"  PCA {data.x.shape[1]}→{reduce_dim} features "
              f"({var_kept:.1f}% variance retained)")

    data = data.to(DEVICE)
    print(f"  {source} Cora  |  Nodes: {data.num_nodes}  |  "
          f"Edges: {data.num_edges}  |  Features: {data.num_node_features}  "
          f"|  Classes: {num_classes}")
    return data, num_classes


def _synthetic_cora():
    """Generate a synthetic citation graph mimicking Cora's properties."""
    N, D, C = 2708, 500, 7
    centers = np.random.randn(C, D) * 0.15
    y = np.zeros(N, dtype=np.int64)
    per = N // C
    for c in range(C):
        s, e = c * per, (c + 1) * per if c < C - 1 else N
        y[s:e] = c
    np.random.shuffle(y)

    x = np.zeros((N, D), dtype=np.float32)
    for i in range(N):
        raw = centers[y[i]] + np.random.randn(D) * 0.8
        x[i] = (raw > np.percentile(raw, 80)).astype(np.float32)

    edges = set()
    for i in range(N):
        n_nb = np.random.randint(2, 6)
        same = np.where(y == y[i])[0]
        diff = np.where(y != y[i])[0]
        n_s = max(1, int(0.6 * n_nb))
        for j in np.random.choice(same, min(n_s, len(same)), replace=False):
            if j != i:
                edges.add((i, int(j))); edges.add((int(j), i))
        for j in np.random.choice(diff, min(n_nb - n_s, len(diff)), replace=False):
            if j != i:
                edges.add((i, int(j))); edges.add((int(j), i))

    ei = np.array(list(edges), dtype=np.int64).T
    masks = {"train": np.zeros(N, bool), "val": np.zeros(N, bool),
             "test": np.zeros(N, bool)}
    for c in range(C):
        idx = np.where(y == c)[0]; np.random.shuffle(idx)
        masks["train"][idx[:20]] = True
        masks["val"][idx[20:92]] = True
        masks["test"][idx[92:235]] = True

    data = Data(
        x=torch.tensor(x, dtype=torch.float32),
        edge_index=torch.tensor(ei, dtype=torch.long),
        y=torch.tensor(y, dtype=torch.long),
        train_mask=torch.tensor(masks["train"]),
        val_mask=torch.tensor(masks["val"]),
        test_mask=torch.tensor(masks["test"]),
    )
    return data, C


# ═══════════════════════════════════════════════════════════════════════════════
# §2  NEUTROSOPHIC FUZZIFICATION LAYER
# ═══════════════════════════════════════════════════════════════════════════════

class NeutrosophicFuzzification(nn.Module):
    """
    Gaussian-based neutrosophic fuzzification for node features.

    For each feature dimension d, given per-feature statistics μ_d, σ_d
    estimated from training nodes:

        T_d(x) = exp( -(x_d - μ_d)² / (2 σ_d²) )
        F_d(x) = 1 - exp( -(x_d - μ_d)² / (2 (α σ_d)²) )
        I_d(x) = 1 - exp( -(x_d - μ_d)² / (2 (β σ_d)²) )

    α and β are *learnable* scale factors controlling the width of the
    Falsity and Indeterminacy Gaussians respectively.

    Parameters
    ----------
    dim : int       Feature dimensionality.
    alpha : float   Initial value of α (controls F width).
    beta : float    Initial value of β (controls I width).
    """

    def __init__(self, dim: int, alpha: float = 1.5, beta: float = 2.0):
        super().__init__()
        self.log_alpha = nn.Parameter(torch.tensor(float(np.log(alpha))))
        self.log_beta  = nn.Parameter(torch.tensor(float(np.log(beta))))
        self.register_buffer("mu",  torch.zeros(dim))
        self.register_buffer("sig", torch.ones(dim))

    def fit(self, x: torch.Tensor):
        """Estimate μ and σ from the training partition."""
        self.mu.copy_(x.mean(dim=0))
        self.sig.copy_(x.std(dim=0).clamp(min=1e-6))

    @property
    def alpha(self):
        return self.log_alpha.exp()

    @property
    def beta(self):
        return self.log_beta.exp()

    def forward(self, x: torch.Tensor):
        """
        Returns
        -------
        T, I, F : each [N, D]  neutrosophic membership values
        """
        diff_sq = (x - self.mu) ** 2
        denom = 2.0 * self.sig ** 2
        T  = torch.exp(-diff_sq / denom)
        Fv = 1.0 - torch.exp(-diff_sq / (denom * self.alpha ** 2))
        Iv = 1.0 - torch.exp(-diff_sq / (denom * self.beta ** 2))
        return T, Iv, Fv


class EdgeNeutrosophicFuzzification(nn.Module):
    """
    Derive neutrosophic triplets for edges from endpoint features.

    Uses cosine similarity between node feature vectors mapped to [0, 1]:
        sim = (cos(x_src, x_dst) + 1) / 2

        T_e = sim                          (high similarity ⟹ high truth)
        I_e = 1 - |sim - 0.5| * 2         (peaks at ambiguous similarity)
        F_e = 1 - sim                      (high dissimilarity ⟹ high falsity)
    """

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor):
        cos = F.cosine_similarity(x[edge_index[0]], x[edge_index[1]], dim=1)
        sim = (cos + 1.0) / 2.0
        return sim, 1.0 - (sim - 0.5).abs() * 2.0, 1.0 - sim


# ═══════════════════════════════════════════════════════════════════════════════
# §3  FIRST-ORDER TSK RULE LAYER
# ═══════════════════════════════════════════════════════════════════════════════

class TSKRuleAggregation(nn.Module):
    """
    First-order Takagi–Sugeno–Kang (TSK) rule layer for message passing.

    For each edge (u → v) and each of K rules:
      Antecedent:   r_k = σ( W_a · [h_v || h_u || edge_triplet] )
      Consequent:   y_k = W_k · [h_v || h_u || edge_triplet] + b_k

    Normalisation:  r̂_k = softmax(r_k)  (across rules)
    Defuzzification: ŷ = Σ_k  r̂_k · y_k
    Aggregation:    scatter_add over dst, then degree-normalise

    Parameters
    ----------
    d_in  : int   Node feature dimension (after projection of T||I||F).
    d_out : int   Output dimension.
    K     : int   Number of TSK rules.
    """

    def __init__(self, d_in: int, d_out: int, K: int = 3):
        super().__init__()
        self.K = K
        self.proj = nn.Linear(3 * d_in, d_out)
        rule_in = 2 * d_out + 3   # h_v || h_u || (T_e, I_e, F_e)
        self.antecedent  = nn.Linear(rule_in, K)
        self.consequents = nn.ModuleList(
            [nn.Linear(rule_in, d_out) for _ in range(K)]
        )

    def forward(self, T_v, I_v, F_v, edge_index, T_e, I_e, F_e, num_nodes):
        # Project concatenated neutrosophic node features
        h = self.proj(torch.cat([T_v, I_v, F_v], dim=1))       # [N, d_out]
        h_src, h_dst = h[edge_index[0]], h[edge_index[1]]      # [E, d_out]
        edge_feat = torch.stack([T_e, I_e, F_e], dim=1)        # [E, 3]
        rule_in = torch.cat([h_dst, h_src, edge_feat], dim=1)  # [E, 2d+3]

        # Firing strengths  →  normalised across rules
        firing = F.softmax(self.antecedent(rule_in), dim=1)     # [E, K]

        # Defuzzified edge messages (memory-efficient accumulation)
        msg = sum(
            firing[:, k:k+1] * self.consequents[k](rule_in)
            for k in range(self.K)
        )                                                       # [E, d_out]

        # Aggregate to target nodes
        out = torch.zeros(num_nodes, msg.shape[1], device=msg.device)
        out.scatter_add_(0, edge_index[1].unsqueeze(1).expand_as(msg), msg)
        deg = degree(edge_index[1], num_nodes).clamp(min=1).unsqueeze(1)
        return out / deg, firing


class TSKRuleAggregationNoEdge(nn.Module):
    """TSK rule layer variant that ignores edge neutrosophic features."""

    def __init__(self, d_in: int, d_out: int, K: int = 3):
        super().__init__()
        self.K = K
        self.proj = nn.Linear(3 * d_in, d_out)
        rule_in = 2 * d_out
        self.antecedent  = nn.Linear(rule_in, K)
        self.consequents = nn.ModuleList(
            [nn.Linear(rule_in, d_out) for _ in range(K)]
        )

    def forward(self, T_v, I_v, F_v, edge_index, num_nodes):
        h = self.proj(torch.cat([T_v, I_v, F_v], dim=1))
        rule_in = torch.cat([h[edge_index[1]], h[edge_index[0]]], dim=1)
        firing = F.softmax(self.antecedent(rule_in), dim=1)
        msg = sum(
            firing[:, k:k+1] * self.consequents[k](rule_in)
            for k in range(self.K)
        )
        out = torch.zeros(num_nodes, msg.shape[1], device=msg.device)
        out.scatter_add_(0, edge_index[1].unsqueeze(1).expand_as(msg), msg)
        deg = degree(edge_index[1], num_nodes).clamp(min=1).unsqueeze(1)
        return out / deg, firing


# ═══════════════════════════════════════════════════════════════════════════════
# §4  N-GNN LAYER  (Fuzzification → TSK → Defuzzification + Residual)
# ═══════════════════════════════════════════════════════════════════════════════

class NGNNLayer(nn.Module):
    """
    Single N-GNN layer combining fuzzification, TSK rule aggregation,
    defuzzification, residual connection, and dropout.

    H^(l) = σ( TSK_defuzz(Fuzz(H^(l-1)), A) + W_res · H^(l-1) )
    """

    def __init__(self, d_in, d_out, K=3, alpha=1.5, beta=2.0,
                 dropout=0.5, use_edge=True):
        super().__init__()
        self.fuzz = NeutrosophicFuzzification(d_in, alpha, beta)
        self.use_edge = use_edge
        if use_edge:
            self.efuzz = EdgeNeutrosophicFuzzification()
            self.tsk   = TSKRuleAggregation(d_in, d_out, K)
        else:
            self.tsk = TSKRuleAggregationNoEdge(d_in, d_out, K)
        self.residual = (nn.Linear(d_in, d_out)
                         if d_in != d_out else nn.Identity())
        self.dropout = dropout

    def forward(self, x, edge_index):
        T, I, Fv = self.fuzz(x)
        if self.use_edge:
            Te, Ie, Fe = self.efuzz(x, edge_index)
            h, firing = self.tsk(T, I, Fv, edge_index, Te, Ie, Fe, x.shape[0])
        else:
            h, firing = self.tsk(T, I, Fv, edge_index, x.shape[0])
        h = F.relu(h + self.residual(x))
        if self.dropout > 0 and self.training:
            h = F.dropout(h, p=self.dropout)
        return h, firing


# ═══════════════════════════════════════════════════════════════════════════════
# §5  FULL N-GNN MODEL
# ═══════════════════════════════════════════════════════════════════════════════

class NGNN(nn.Module):
    """
    Multi-layer Neutrosophic Graph Neural Network.

    Stacks L NGNNLayers with residual connections, ending in a
    log-softmax classification head.

    Parameters
    ----------
    d_in    : int    Input feature dimension.
    d_hid   : int    Hidden layer dimension.
    n_cls   : int    Number of output classes.
    K       : int    Number of TSK rules per layer.
    alpha   : float  Initial α for fuzzification.
    beta    : float  Initial β for fuzzification.
    dropout : float  Dropout rate.
    use_edge: bool   Whether to use edge neutrosophic features.
    """

    def __init__(self, d_in, d_hid, n_cls, K=3, alpha=1.5, beta=2.0,
                 dropout=0.5, use_edge=True):
        super().__init__()
        self.layer1 = NGNNLayer(d_in, d_hid, K, alpha, beta, dropout, use_edge)
        self.layer2 = NGNNLayer(d_hid, n_cls, K, alpha, beta, 0.0, use_edge)

    def fit_fuzz(self, x_train: torch.Tensor):
        """Fit fuzzification statistics on training node features."""
        self.layer1.fuzz.fit(x_train)

    def forward(self, x, edge_index):
        x, f1 = self.layer1(x, edge_index)
        x, f2 = self.layer2(x, edge_index)
        return F.log_softmax(x, dim=1), [f1, f2]


# ═══════════════════════════════════════════════════════════════════════════════
# §6  BASELINE: STANDARD GCN
# ═══════════════════════════════════════════════════════════════════════════════

class GCN(nn.Module):
    """Two-layer Graph Convolutional Network (Kipf & Welling, 2017)."""

    def __init__(self, d_in, d_hid, n_cls, dropout=0.5):
        super().__init__()
        self.w1 = nn.Linear(d_in, d_hid)
        self.w2 = nn.Linear(d_hid, n_cls)
        self.dropout = dropout

    def _propagate(self, x, W, edge_index):
        ei, _ = add_self_loops(edge_index, num_nodes=x.size(0))
        s, d = ei
        dg = degree(d, x.size(0)).clamp(min=1)
        norm = dg[s].pow(-0.5) * dg[d].pow(-0.5)
        h = W(x)
        msg = h[s] * norm.unsqueeze(1)
        out = torch.zeros_like(h)
        out.scatter_add_(0, d.unsqueeze(1).expand_as(msg), msg)
        return out

    def forward(self, x, edge_index):
        x = F.relu(self._propagate(x, self.w1, edge_index))
        if self.training:
            x = F.dropout(x, p=self.dropout)
        x = self._propagate(x, self.w2, edge_index)
        return F.log_softmax(x, dim=1), []

    def fit_fuzz(self, x):
        pass  # no-op for interface compatibility


# ═══════════════════════════════════════════════════════════════════════════════
# §7  ABLATION VARIANT: N-GNN WITHOUT FUZZIFICATION
# ═══════════════════════════════════════════════════════════════════════════════

class NGNNNoFuzz(nn.Module):
    """
    N-GNN variant that skips the fuzzification step.
    Uses raw features with TSK-style aggregation (no T/I/F encoding).
    """

    def __init__(self, d_in, d_hid, n_cls, K=3, dropout=0.5):
        super().__init__()
        self.K = K
        self.p1 = nn.Linear(d_in, d_hid)
        self.a1 = nn.Linear(2 * d_hid, K)
        self.c1 = nn.ModuleList([nn.Linear(2 * d_hid, d_hid) for _ in range(K)])
        self.r1 = nn.Linear(d_in, d_hid) if d_in != d_hid else nn.Identity()
        self.p2 = nn.Linear(d_hid, n_cls)
        self.a2 = nn.Linear(2 * n_cls, K)
        self.c2 = nn.ModuleList([nn.Linear(2 * n_cls, n_cls) for _ in range(K)])
        self.r2 = nn.Linear(d_hid, n_cls) if d_hid != n_cls else nn.Identity()
        self.dropout = dropout

    def _layer(self, x, proj, ante, cons, res, ei, drop):
        h = proj(x)
        ri = torch.cat([h[ei[1]], h[ei[0]]], 1)
        fn = F.softmax(ante(ri), 1)
        msg = sum(fn[:, k:k+1] * cons[k](ri) for k in range(self.K))
        out = torch.zeros(x.shape[0], msg.shape[1], device=x.device)
        out.scatter_add_(0, ei[1].unsqueeze(1).expand_as(msg), msg)
        dg = degree(ei[1], x.shape[0]).clamp(min=1).unsqueeze(1)
        out = F.relu(out / dg + res(x))
        if drop > 0 and self.training:
            out = F.dropout(out, p=drop)
        return out

    def fit_fuzz(self, x):
        pass

    def forward(self, x, ei):
        x = self._layer(x, self.p1, self.a1, self.c1, self.r1, ei, self.dropout)
        x = self._layer(x, self.p2, self.a2, self.c2, self.r2, ei, 0.0)
        return F.log_softmax(x, dim=1), []


# ═══════════════════════════════════════════════════════════════════════════════
# §8  TRAINING ENGINE
# ═══════════════════════════════════════════════════════════════════════════════

def train_and_evaluate(model, data, epochs=80, lr=0.01, weight_decay=5e-4,
                       verbose=True):
    """
    Train a model with early stopping on validation accuracy.

    Returns
    -------
    model     : trained model (best checkpoint)
    history   : dict of lists {train_loss, train_acc, val_acc, test_acc}
    test_acc  : float
    test_f1   : float (macro)
    pred      : Tensor of predicted labels
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=lr,
                                 weight_decay=weight_decay)
    history = {"train_loss": [], "train_acc": [], "val_acc": [], "test_acc": []}
    best_val, best_state = 0.0, None

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        out, _ = model(data.x, data.edge_index)
        loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            out, _ = model(data.x, data.edge_index)
            pred = out.argmax(dim=1)
            ta = (pred[data.train_mask] == data.y[data.train_mask]).float().mean().item()
            va = (pred[data.val_mask]   == data.y[data.val_mask]).float().mean().item()
            te = (pred[data.test_mask]  == data.y[data.test_mask]).float().mean().item()

        history["train_loss"].append(loss.item())
        history["train_acc"].append(ta)
        history["val_acc"].append(va)
        history["test_acc"].append(te)

        if va > best_val:
            best_val = va
            best_state = copy.deepcopy(model.state_dict())

        if verbose and epoch % 20 == 0:
            print(f"  Epoch {epoch:3d}  |  Loss {loss.item():.4f}  |  "
                  f"Train {ta:.4f}  |  Val {va:.4f}  |  Test {te:.4f}")

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        out, _ = model(data.x, data.edge_index)
        pred = out.argmax(dim=1)
        test_acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()
        test_f1 = f1_score(data.y[data.test_mask].cpu().numpy(),
                           pred[data.test_mask].cpu().numpy(),
                           average="macro")
    return model, history, test_acc, test_f1, pred


# ═══════════════════════════════════════════════════════════════════════════════
# §9  ABLATION STUDY
# ═══════════════════════════════════════════════════════════════════════════════

def run_ablation(data, D, C, epochs=50):
    """Compare Standard GCN vs. N-GNN variants."""
    configs = {
        "Standard GCN":       lambda: GCN(D, 32, C),
        "N-GNN (full)":       lambda: NGNN(D, 32, C, K=3, use_edge=True),
        "N-GNN (no edge)":    lambda: NGNN(D, 32, C, K=3, use_edge=False),
        "N-GNN (no fuzz)":    lambda: NGNNNoFuzz(D, 32, C, K=3),
    }
    results, histories = {}, {}
    for name, builder in configs.items():
        print(f"\n{'─'*50}\n  Training: {name}\n{'─'*50}")
        model = builder().to(DEVICE)
        model.fit_fuzz(data.x[data.train_mask])
        model, hist, acc, f1, _ = train_and_evaluate(model, data, epochs=epochs)
        n_params = sum(p.numel() for p in model.parameters())
        results[name] = {"acc": acc, "f1": f1, "params": n_params}
        histories[name] = hist
        print(f"  ▸ {name}  Acc={acc:.4f}  F1={f1:.4f}  Params={n_params:,}")
        del model; gc.collect()
    return results, histories


# ═══════════════════════════════════════════════════════════════════════════════
# §10  HYPERPARAMETER SENSITIVITY (α, β)
# ═══════════════════════════════════════════════════════════════════════════════

def alpha_beta_grid(data, D, C, epochs=40):
    """Sweep over α and β to study their effect on accuracy."""
    alphas = [0.5, 1.5, 3.0]
    betas  = [0.5, 1.5, 3.0]
    rows = []
    for a in alphas:
        for b in betas:
            m = NGNN(D, 32, C, K=3, alpha=a, beta=b).to(DEVICE)
            m.fit_fuzz(data.x[data.train_mask])
            _, _, acc, f1, _ = train_and_evaluate(m, data, epochs=epochs,
                                                   verbose=False)
            rows.append({"alpha": a, "beta": b, "acc": acc, "f1": f1})
            print(f"  α={a:.1f}  β={b:.1f}  →  Acc={acc:.4f}")
            del m; gc.collect()
    return pd.DataFrame(rows)


# ═══════════════════════════════════════════════════════════════════════════════
# §11  VISUALISATION UTILITIES
# ═══════════════════════════════════════════════════════════════════════════════

def plot_architecture(path):
    """Generate a schematic diagram of the N-GNN pipeline."""
    fig, ax = plt.subplots(figsize=(16, 6))
    ax.set_xlim(0, 16); ax.set_ylim(0, 6); ax.axis("off")

    boxes = [
        (0.5, 2, 2.5, 2,
         "Input Graph\nG = (V, E)\nNode features X\nEdge index",
         "#E3F2FD"),
        (3.5, 2, 2.8, 2,
         "Neutrosophic\nFuzzification\n─────────\nT = exp(−(x−μ)²/2σ²)\n"
         "F = 1−exp(−(x−μ)²/2(ασ)²)\nI = 1−exp(−(x−μ)²/2(βσ)²)",
         "#FFF9C4"),
        (7.0, 2, 2.8, 2,
         "TSK Rule Layer\n─────────\nr_k = Comb(N_V, N_E)\n"
         "ŷ_k = f_k(T,I,F)\n\nNormalisation:\nr̂_k = r_k / Σ r_j",
         "#FFCCBC"),
        (10.5, 2, 2.5, 2,
         "Defuzzification\n─────────\ny = Σ r̂_k · f_k(x)\n"
         "+ Residual\n+ ReLU",
         "#C8E6C9"),
        (13.5, 2, 2.0, 2,
         "Output\n─────────\nSoftmax\nNode class\nprediction",
         "#E1BEE7"),
    ]
    for (bx, by, bw, bh, txt, col) in boxes:
        ax.add_patch(plt.Rectangle((bx, by), bw, bh, facecolor=col,
                                    edgecolor="#333", lw=1.5))
        ax.text(bx + bw / 2, by + bh / 2, txt, ha="center", va="center",
                fontsize=7.5, fontfamily="monospace")
    for x1, x2 in [(3.0, 3.5), (6.3, 7.0), (9.8, 10.5), (13.0, 13.5)]:
        ax.annotate("", xy=(x2, 3), xytext=(x1, 3),
                     arrowprops=dict(arrowstyle="->", lw=2, color="#555"))
    ax.text(8, 5.2,
            "Neutrosophic Graph Neural Network (N-GNN) — Architecture",
            ha="center", fontsize=15, fontweight="bold")
    ax.text(8, 0.6,
            "↻ Stack L layers with residual:  "
            "H⁽ˡ⁾ = σ( f⁽ˡ⁾(H⁽ˡ⁻¹⁾, A) + H⁽ˡ⁻¹⁾ )",
            ha="center", fontsize=10, style="italic", color="#555")
    plt.savefig(path, dpi=150, bbox_inches="tight"); plt.close()


def plot_fuzzification(data, path, alpha=1.5, beta=2.0):
    """Show Gaussian T/I/F membership functions for top-variance features."""
    x_np = data.x.cpu().numpy()
    mu, sig = x_np.mean(0), x_np.std(0) + 1e-6
    top = np.argsort(-sig)[:3]

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    for i, fi in enumerate(top):
        xs = np.linspace(x_np[:, fi].min() - 0.5,
                         x_np[:, fi].max() + 0.5, 300)
        m, s = mu[fi], sig[fi]
        T  = np.exp(-((xs - m) ** 2) / (2 * s ** 2))
        Fv = 1 - np.exp(-((xs - m) ** 2) / (2 * (alpha * s) ** 2))
        Iv = 1 - np.exp(-((xs - m) ** 2) / (2 * (beta * s) ** 2))
        axes[i].plot(xs, T,  "g-",  lw=2, label="T (Truth)")
        axes[i].plot(xs, Iv, "b--", lw=2, label="I (Indeterminacy)")
        axes[i].plot(xs, Fv, "r-.", lw=2, label="F (Falsity)")
        axes[i].set_title(f"Feature {fi}", fontsize=12)
        axes[i].set_xlabel("Feature value")
        axes[i].set_ylabel("Membership degree")
        axes[i].legend(fontsize=9)
        axes[i].set_ylim(-0.05, 1.05)
    fig.suptitle("Neutrosophic Fuzzification (Gaussian-based T, I, F)",
                 fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight"); plt.close()


def plot_training_curves(histories, path):
    """Plot training loss, train accuracy, and test accuracy."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for name, h in histories.items():
        axes[0].plot(h["train_loss"], label=name, lw=1.2)
        axes[1].plot(h["train_acc"],  label=name, lw=1.2)
        axes[2].plot(h["test_acc"],   label=name, lw=1.2)
    titles = ["Training Loss", "Training Accuracy", "Test Accuracy"]
    for ax, t in zip(axes, titles):
        ax.set_title(t, fontsize=13); ax.set_xlabel("Epoch")
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight"); plt.close()


def plot_ablation_bar(results, path):
    """Bar chart comparing ablation results."""
    names = list(results.keys())
    accs = [results[n]["acc"] for n in names]
    f1s  = [results[n]["f1"]  for n in names]
    x = np.arange(len(names)); w = 0.35

    fig, ax = plt.subplots(figsize=(10, 5))
    b1 = ax.bar(x - w / 2, accs, w, label="Accuracy", color="#42A5F5")
    b2 = ax.bar(x + w / 2, f1s,  w, label="Macro-F1", color="#66BB6A")
    for bars in [b1, b2]:
        for b in bars:
            ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.005,
                    f"{b.get_height():.4f}", ha="center", fontsize=9)
    ax.set_xticks(x); ax.set_xticklabels(names, fontsize=9)
    ax.set_title("Ablation Study: N-GNN Variants on Cora", fontsize=14)
    ax.legend(); ax.set_ylim(0, 1.05)
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight"); plt.close()


def plot_ab_heatmap(df, path):
    """Heatmap of test accuracy across α × β grid."""
    pivot = df.pivot(index="beta", columns="alpha", values="acc")
    fig, ax = plt.subplots(figsize=(7, 5.5))
    im = ax.imshow(pivot.values, cmap="YlOrRd", aspect="auto", origin="lower")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f"{v:.1f}" for v in pivot.columns])
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f"{v:.1f}" for v in pivot.index])
    ax.set_xlabel("α (Falsity width)", fontsize=12)
    ax.set_ylabel("β (Indeterminacy width)", fontsize=12)
    ax.set_title("Test Accuracy vs. α and β", fontsize=14)
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            ax.text(j, i, f"{pivot.values[i, j]:.3f}",
                    ha="center", va="center", fontsize=9)
    plt.colorbar(im, ax=ax, label="Test Accuracy")
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight"); plt.close()


# ═══════════════════════════════════════════════════════════════════════════════
# §12  MAIN PIPELINE
# ═══════════════════════════════════════════════════════════════════════════════

def main():
    print("=" * 70)
    print("  N-GNN: Neutrosophic Graph Neural Network — Node Classification")
    print(f"  Device: {DEVICE}   Output: {OUT_DIR}")
    print("=" * 70)

    # ── Load data ──
    data, C = load_cora(reduce_dim=64)
    D = data.num_node_features

    # ── [1/5] Architecture diagram ──
    print("\n[1/5] Architecture diagram...")
    plot_architecture(os.path.join(OUT_DIR, "ngnn_architecture.png"))
    print("  ✓ Saved")

    # ── [2/5] Fuzzification visualisation ──
    print("\n[2/5] Fuzzification visualisation...")
    plot_fuzzification(data, os.path.join(OUT_DIR, "ngnn_fuzzification.png"))
    print("  ✓ Saved")

    # ── [3/5] Ablation study ──
    print("\n[3/5] Ablation study...")
    results, histories = run_ablation(data, D, C, epochs=50)
    print("\n" + "=" * 60)
    print("  ABLATION RESULTS")
    print("=" * 60)
    for name, r in results.items():
        print(f"  {name:25s}  Acc={r['acc']:.4f}  "
              f"F1={r['f1']:.4f}  Params={r['params']:,}")

    plot_training_curves(histories,
                         os.path.join(OUT_DIR, "ngnn_training_curves.png"))
    plot_ablation_bar(results,
                      os.path.join(OUT_DIR, "ngnn_ablation_bar.png"))
    print("  ✓ Plots saved")

    # ── [4/5] α/β hyperparameter analysis ──
    print("\n[4/5] α/β hyperparameter analysis...")
    ab_df = alpha_beta_grid(data, D, C, epochs=40)
    plot_ab_heatmap(ab_df, os.path.join(OUT_DIR, "ngnn_alpha_beta_heatmap.png"))
    best = ab_df.loc[ab_df["acc"].idxmax()]
    print(f"  Best: α={best['alpha']:.1f}  β={best['beta']:.1f}  "
          f"Acc={best['acc']:.4f}")

    # ── [5/5] Final evaluation ──
    print("\n[5/5] Final evaluation with best hyperparameters...")
    model = NGNN(D, 32, C, K=3,
                 alpha=float(best["alpha"]),
                 beta=float(best["beta"])).to(DEVICE)
    model.fit_fuzz(data.x[data.train_mask])
    model, _, acc, f1, pred = train_and_evaluate(model, data, epochs=50)

    yt = data.y[data.test_mask].cpu().numpy()
    yp = pred[data.test_mask].cpu().numpy()
    class_names = ["Case_Based", "Genetic_Alg", "Neural_Nets",
                   "Prob_Methods", "Reinforc_Learn", "Rule_Learning", "Theory"]
    print("\n  Classification Report (Test Set):")
    print(classification_report(yt, yp, target_names=class_names,
                                 digits=4, zero_division=0))

    # ── Save summary ──
    summary = {
        "ablation": results,
        "best_alpha": float(best["alpha"]),
        "best_beta": float(best["beta"]),
        "final_test_acc": acc,
        "final_macro_f1": f1,
        "alpha_beta_grid": ab_df.to_dict(orient="records"),
    }
    with open(os.path.join(OUT_DIR, "ngnn_summary.json"), "w") as f:
        json.dump(summary, f, indent=2, default=str)

    print("\n" + "=" * 70)
    print(f"  All outputs saved to {OUT_DIR}/")
    print("=" * 70)
    return results, ab_df


if __name__ == "__main__":
    main()

  N-GNN: Neutrosophic Graph Neural Network — Node Classification
  Device: cpu   Output: /Users/rajshrestha/Library/Group Containers/group.com.apple.reminders/Container_v1/MLModels/Layers/outputs


Processing...
Done!


  PCA 64→64 features (34.9% variance retained)
  Planetoid Cora  |  Nodes: 2708  |  Edges: 10556  |  Features: 64  |  Classes: 7

[1/5] Architecture diagram...
  ✓ Saved

[2/5] Fuzzification visualisation...
  ✓ Saved

[3/5] Ablation study...

──────────────────────────────────────────────────
  Training: Standard GCN
──────────────────────────────────────────────────
  Epoch  20  |  Loss 0.9800  |  Train 0.8714  |  Val 0.6960  |  Test 0.6940
  Epoch  40  |  Loss 0.2585  |  Train 0.9571  |  Val 0.7780  |  Test 0.7980
  ▸ Standard GCN  Acc=0.7980  F1=0.7913  Params=2,311

──────────────────────────────────────────────────
  Training: N-GNN (full)
──────────────────────────────────────────────────
  Epoch  20  |  Loss 1.3320  |  Train 0.5500  |  Val 0.3820  |  Test 0.3940
  Epoch  40  |  Loss 0.8805  |  Train 0.7143  |  Val 0.4620  |  Test 0.4580
  ▸ N-GNN (full)  Acc=0.4650  F1=0.3405  Params=16,334

──────────────────────────────────────────────────
  Training: N-GNN (no edge)
────────

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.utils import add_self_loops, degree

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ─── Load Cora ──────────────────────────────────────────
dataset = Planetoid(root="./data", name="Cora")
data = dataset[0].to(DEVICE)
num_features = data.num_node_features
num_classes = dataset.num_classes

# ─── Neutrosophic Fuzzification ─────────────────────────
class NeutrosophicFuzz(nn.Module):
    def __init__(self, dim, alpha=1.5, beta=2.0):
        super().__init__()
        self.log_alpha = nn.Parameter(torch.log(torch.tensor(alpha)))
        self.log_beta  = nn.Parameter(torch.log(torch.tensor(beta)))
        self.register_buffer("mu", torch.zeros(dim))
        self.register_buffer("sig", torch.ones(dim))

    @property
    def alpha(self): return self.log_alpha.exp()
    @property
    def beta(self):  return self.log_beta.exp()

    def fit(self, x):
        self.mu.copy_(x.mean(0))
        self.sig.copy_(x.std(0).clamp(min=1e-6))

    def forward(self, x):
        d = (x - self.mu)**2
        denom = 2*self.sig**2
        T = torch.exp(-d / denom)
        Fv = 1 - torch.exp(-d / (denom * self.alpha**2))
        I = 1 - torch.exp(-d / (denom * self.beta**2))
        return T, I, Fv

# ─── TSK Rule Layer ─────────────────────────────────────
class TSKLayer(nn.Module):
    def __init__(self, d_in, d_out, K=3):
        super().__init__()
        self.K = K
        self.proj = nn.Linear(3*d_in, d_out)
        self.ante = nn.Linear(2*d_out, K)
        self.cons = nn.ModuleList([nn.Linear(2*d_out, d_out) for _ in range(K)])

    def forward(self, T, I, Fv, edge_index):
        h = self.proj(torch.cat([T,I,Fv],1))
        h_src, h_dst = h[edge_index[0]], h[edge_index[1]]
        ri = torch.cat([h_dst, h_src],1)
        firing = F.softmax(self.ante(ri),1)
        msg = sum(firing[:,k:k+1]*self.cons[k](ri) for k in range(self.K))
        out = torch.zeros_like(h)
        out.scatter_add_(0, edge_index[1].unsqueeze(1).expand_as(msg), msg)
        deg = degree(edge_index[1], h.size(0)).clamp(min=1).unsqueeze(1)
        return out / deg

# ─── NGNN Layer ─────────────────────────────────────────
class NGNNLayer(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.fuzz = NeutrosophicFuzz(d_in)
        self.tsk  = TSKLayer(d_in, d_out)
        self.res  = nn.Linear(d_in,d_out) if d_in!=d_out else nn.Identity()

    def forward(self, x, edge_index):
        T,I,Fv = self.fuzz(x)
        h = self.tsk(T,I,Fv,edge_index)
        return F.relu(h + self.res(x))

# ─── Full NGNN ─────────────────────────────────────────
class NGNN(nn.Module):
    def __init__(self, d_in, d_hid, n_cls):
        super().__init__()
        self.l1 = NGNNLayer(d_in, d_hid)
        self.l2 = NGNNLayer(d_hid, n_cls)

    def fit_fuzz(self, x): self.l1.fuzz.fit(x)
    def forward(self, x, edge_index):
        x = self.l1(x, edge_index)
        x = self.l2(x, edge_index)
        return F.log_softmax(x,1)

# ─── Train & Evaluate ──────────────────────────────────
# def train(model, data, epochs=100, lr=0.01):
#     optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
#     best_val, best_state = 0, None
#     model.fit_fuzz(data.x[data.train_mask])
#     for _ in range(epochs):
#         model.train()
#         optimizer.zero_grad()
#         out = model(data.x, data.edge_index)
#         loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
#         loss.backward(); optimizer.step()
#         model.eval()
#         with torch.no_grad():
#             pred = out.argmax(1)
#             val_acc = (pred[data.val_mask]==data.y[data.val_mask]).float().mean().item()
#             if val_acc>best_val:
#                 best_val = val_acc
#                 best_state = model.state_dict()
#     model.load_state_dict(best_state)
#     model.eval()
#     with torch.no_grad():
#         out = model(data.x,data.edge_index)
#         pred = out.argmax(1)
#         test_acc = (pred[data.test_mask]==data.y[data.test_mask]).float().item()
#     return model, test_acc
def train(model, data, epochs=100, lr=0.01):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    best_val, best_state = 0, None
    model.fit_fuzz(data.x[data.train_mask])
    for _ in range(epochs):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()
        
        model.eval()
        with torch.no_grad():
            pred = out.argmax(1)
            val_acc = (pred[data.val_mask]==data.y[data.val_mask]).float().mean().item()
            if val_acc>best_val:
                best_val = val_acc
                best_state = model.state_dict()
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        out = model(data.x,data.edge_index)
        pred = out.argmax(1)
        test_acc = (pred[data.test_mask]==data.y[data.test_mask]).float().mean().item()
    return model, test_acc
# ─── Run ───────────────────────────────────────────────
model = NGNN(num_features, 32, num_classes).to(DEVICE)
model, test_acc = train(model, data)
print("Test Accuracy:", test_acc)

Test Accuracy: 0.12999999523162842


In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv
import matplotlib.pyplot as plt

# ─── Device ───────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ─── Load Cora dataset ───────────────────
dataset = Planetoid(root='data/Cora', name='Cora')
data = dataset[0].to(DEVICE)
num_features = dataset.num_node_features
num_classes = dataset.num_classes

# ─── Simple NGNN-like GCN Model ──────────
class NGNN(nn.Module):
    def __init__(self, in_feats, hidden, out_feats):
        super().__init__()
        self.conv1 = GCNConv(in_feats, hidden)
        self.conv2 = GCNConv(hidden, out_feats)
    
    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = torch.tanh(x)  # Neutrosophic-like activation
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)

# ─── Training ────────────────────────────
def train(model, data, epochs=200, lr=0.01):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    model.train()
    train_accs, val_accs, losses = [], [], []

    best_val, best_state = 0, None
    for epoch in range(1, epochs+1):
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()
        
        # Evaluation
        model.eval()
        with torch.no_grad():
            pred = out.argmax(1)
            train_acc = (pred[data.train_mask]==data.y[data.train_mask]).float().mean().item()
            val_acc = (pred[data.val_mask]==data.y[data.val_mask]).float().mean().item()
            if val_acc > best_val:
                best_val = val_acc
                best_state = model.state_dict()
        train_accs.append(train_acc)
        val_accs.append(val_acc)
        losses.append(loss.item())
        model.train()
        
        if epoch % 20 == 0:
            print(f"Epoch {epoch} | Loss: {loss.item():.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")
    
    model.load_state_dict(best_state)
    return model, losses, train_accs, val_accs

# ─── Run training ────────────────────────
model = NGNN(num_features, 32, num_classes).to(DEVICE)
model, losses, train_accs, val_accs = train(model, data)

# ─── Test accuracy ───────────────────────
model.eval()
with torch.no_grad():
    out = model(data.x, data.edge_index)
    pred = out.argmax(1)
    test_acc = (pred[data.test_mask]==data.y[data.test_mask]).float().mean().item()
print("Final Test Accuracy:", test_acc)

# ─── Plot training curve ─────────────────
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(losses, label='Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('Training Loss'); plt.legend()
plt.subplot(1,2,2)
plt.plot(train_accs, label='Train Acc')
plt.plot(val_accs, label='Val Acc')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.title('Training Accuracy'); plt.legend()
plt.show()

# ─── Visualize node classification ───────
plt.figure(figsize=(6,6))
plt.scatter(range(data.num_nodes), [0]*data.num_nodes, c=pred.cpu(), cmap='tab20', s=50)
plt.title("Node Classification Results")
plt.xlabel("Node index")
plt.yticks([])
plt.show()

Epoch 20 | Loss: 0.0607 | Train Acc: 1.0000 | Val Acc: 0.7720
Epoch 40 | Loss: 0.0130 | Train Acc: 1.0000 | Val Acc: 0.7740
Epoch 60 | Loss: 0.0163 | Train Acc: 1.0000 | Val Acc: 0.7800
Epoch 80 | Loss: 0.0208 | Train Acc: 1.0000 | Val Acc: 0.7600
Epoch 100 | Loss: 0.0181 | Train Acc: 1.0000 | Val Acc: 0.7600
Epoch 120 | Loss: 0.0142 | Train Acc: 1.0000 | Val Acc: 0.7720
Epoch 140 | Loss: 0.0144 | Train Acc: 1.0000 | Val Acc: 0.7500
Epoch 160 | Loss: 0.0128 | Train Acc: 1.0000 | Val Acc: 0.7700
Epoch 180 | Loss: 0.0120 | Train Acc: 1.0000 | Val Acc: 0.7720
Epoch 200 | Loss: 0.0109 | Train Acc: 1.0000 | Val Acc: 0.7600
Final Test Accuracy: 0.7979999780654907


In [9]:
import networkx as nx
from torch_geometric.utils import to_networkx

# ─── Convert PyG graph to NetworkX ─────────
G = to_networkx(data, to_undirected=True)

# ─── Node colors from predictions ──────────
node_colors = pred.cpu().numpy()

# ─── Plot ──────────────────────────────────
plt.figure(figsize=(10,10))
pos = nx.spring_layout(G, seed=42)  # positions for all nodes
nx.draw_networkx_nodes(G, pos, node_size=100, node_color=node_colors, cmap='tab20')
nx.draw_networkx_edges(G, pos, alpha=0.3)
plt.title("Cora Node Classification (Predicted Classes)")
plt.axis('off')
plt.show()